In [ ]:
import subprocess
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from packaging.version import Version

# Vá thuộc tính pd.util.version để XGBoost tương thích môi trường Kaggle offline
if not hasattr(pd, "util"):
    class _DummyUtil:
        pass
    pd.util = _DummyUtil()
if not hasattr(pd.util, "version"):
    class _DummyVersion:
        Version = Version
    pd.util.version = _DummyVersion()

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from xgboost import XGBClassifier

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/tnghonglm/read-csv"),
        Path("/kaggle/input/playground-series-s6e11"),
        Path("data"),
    ]
    for p in candidates:
        if (p / "train.csv").exists():
            return p
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        found = next(kaggle_root.rglob("train.csv"), None)
        if found is not None:
            return found.parent
    raise FileNotFoundError("Không tìm thấy train.csv trong /kaggle/input hoặc data/")

DATA_DIR = find_data_dir()
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    USE_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    USE_GPU = False

# Trên CPU cá nhân: CatBoost & Neural Net rất chậm nên mặc định chỉ chạy XGB + LGB.
# Khi chạy trên Kaggle GPU: tự động bật cả 4 mô hình (XGB + LGB + CAT + NN).
INCLUDE_CATBOOST = USE_GPU
INCLUDE_NN = USE_GPU and HAS_TORCH
NN_SEEDS = (42, 2024) if USE_GPU else (42,)
print(f"DATA_DIR: {DATA_DIR} | USE_GPU: {USE_GPU} | CAT: {INCLUDE_CATBOOST} | NN: {INCLUDE_NN}")

BOOL_COLS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]

CAT_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
FLOAT_COLS_2D = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]
FLOAT_COLS_1D = ["velocity_score", "merchant_risk_score"]

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Chuyển kiểu boolean sang 0/1, giữ nguyên 24 cột gốc cho GBDT."""
    out = df.copy()
    out[BOOL_COLS] = out[BOOL_COLS].astype(int)
    return out


def _build_digit_keys(df: pd.DataFrame) -> dict[str, np.ndarray]:
    """Tách các cột số thực thành thành phần chữ số (an toàn biên NaN/âm)."""
    keys: dict[str, np.ndarray] = {}
    for col in FLOAT_COLS_2D:
        vals = df[col].to_numpy(dtype=np.float64)
        is_nan = np.isnan(vals)
        safe_raw = np.where(is_nan, 0.0, vals)
        int_part = np.floor(safe_raw + 1e-9).astype(np.int64)
        scaled100 = np.round(safe_raw * 100).astype(np.int64)
        keys[f"{col}__int"] = np.where(is_nan, -1, int_part)
        keys[f"{col}__cents"] = np.where(is_nan, -1, np.abs(scaled100) % 100)
        keys[f"{col}__tenths"] = np.where(is_nan, -1, (np.abs(scaled100) // 10) % 10)
        keys[f"{col}__hundredths"] = np.where(is_nan, -1, np.abs(scaled100) % 10)
        keys[f"{col}__ones"] = np.where(is_nan, -1, np.abs(int_part) % 10)
        keys[f"{col}__tens"] = np.where(is_nan, -1, (np.abs(int_part) // 10) % 10)
        safe_pos = np.where(safe_raw > 0, safe_raw, 1.0)
        exp10 = np.floor(np.log10(safe_pos))
        lead = np.where(safe_raw > 0, np.floor(safe_pos / (10.0 ** exp10) + 1e-9).astype(np.int64), 0)
        keys[f"{col}__lead"] = np.where(is_nan, -1, lead)

    for col in FLOAT_COLS_1D:
        vals = df[col].to_numpy(dtype=np.float64)
        is_nan = np.isnan(vals)
        safe_raw = np.where(is_nan, 0.0, vals)
        int_part = np.floor(safe_raw + 1e-9).astype(np.int64)
        scaled10 = np.round(safe_raw * 10).astype(np.int64)
        keys[f"{col}__int"] = np.where(is_nan, -1, int_part)
        keys[f"{col}__tenths"] = np.where(is_nan, -1, np.abs(scaled10) % 10)
        keys[f"{col}__ones"] = np.where(is_nan, -1, np.abs(int_part) % 10)
        keys[f"{col}__tens"] = np.where(is_nan, -1, (np.abs(int_part) // 10) % 10)
    return keys


def add_fold_digit_te(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    smooth: float = 20.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Cross-fitted Target Encoding trên 36 Digit Keys (chỉ fit từ X_tr/y_tr, nâng 24 -> 60 cột)."""
    tr, val, te = X_tr.copy(), X_val.copy(), X_test.copy()
    tr_keys = _build_digit_keys(tr)
    val_keys = _build_digit_keys(val)
    te_keys = _build_digit_keys(te)

    inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    y_arr = y_tr.to_numpy(dtype=np.float64)
    global_prior = float(y_arr.mean())
    inner_splits = list(inner_skf.split(tr, y_arr))

    for col_name, tr_k in tr_keys.items():
        oof_te = np.full(len(tr), global_prior, dtype=np.float32)
        for itr, ival in inner_splits:
            prior = float(y_arr[itr].mean())
            agg = pd.DataFrame({"k": tr_k[itr], "y": y_arr[itr]}).groupby("k", sort=False)["y"].agg(["sum", "count"])
            enc = (agg["sum"] + smooth * prior) / (agg["count"] + smooth)
            oof_te[ival] = pd.Series(tr_k[ival]).map(enc).fillna(prior).to_numpy(dtype=np.float32)

        tr[f"te_{col_name}"] = oof_te
        full_agg = pd.DataFrame({"k": tr_k, "y": y_arr}).groupby("k", sort=False)["y"].agg(["sum", "count"])
        full_enc = (full_agg["sum"] + smooth * global_prior) / (full_agg["count"] + smooth)
        val[f"te_{col_name}"] = pd.Series(val_keys[col_name]).map(full_enc).fillna(global_prior).to_numpy(dtype=np.float32)
        te[f"te_{col_name}"] = pd.Series(te_keys[col_name]).map(full_enc).fillna(global_prior).to_numpy(dtype=np.float32)
    return tr, val, te


def add_nn_extra_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo thêm đặc trưng tỷ lệ & chu kỳ thời gian riêng cho mạng Neural Network."""
    out = df.copy()
    out["amount_to_balance_ratio"] = out["amount_usd"] / (out["account_balance_usd"].clip(lower=0) + 1.0)
    out["balance_diff"] = out["account_balance_usd"] - out["amount_usd"]
    out["txn_intensity"] = out["txn_count_last_24h"] / (out["hours_since_last_txn"].clip(lower=0) + 0.1)
    out["travel_speed_kmh"] = out["distance_from_home_km"] / (out["hours_since_last_txn"].clip(lower=0) + 0.1)
    out["risk_flag_count"] = out[BOOL_COLS].sum(axis=1)
    out["cvv_plus_disputes"] = out["cvv_retry_count"] + out["prior_disputes"]
    out["log_amount"] = np.log1p(out["amount_usd"].clip(lower=0))
    out["log_distance"] = np.log1p(out["distance_from_home_km"].clip(lower=0))
    out["log_balance"] = np.log1p(out["account_balance_usd"].clip(lower=0))
    out["hour_sin"] = np.sin(2 * np.pi * out["time_of_day_hour"] / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * out["time_of_day_hour"] / 24.0)
    out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7.0)
    return out

In [ ]:
LR = 0.025 if USE_GPU else 0.04
N_EST = 8000 if USE_GPU else 5000
EARLY_STOP = 150 if USE_GPU else 100

def build_xgb() -> XGBClassifier:
    return XGBClassifier(
        n_estimators=N_EST,
        learning_rate=LR,
        max_depth=2,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.70,
        reg_alpha=2.0,
        reg_lambda=10.0,
        max_bin=512,
        tree_method="hist",
        device="cuda" if USE_GPU else "cpu",
        enable_categorical=True,
        eval_metric="auc",
        early_stopping_rounds=EARLY_STOP,
        n_jobs=-1,
        random_state=42,
    )


def build_lgb() -> LGBMClassifier:
    return LGBMClassifier(
        n_estimators=N_EST,
        learning_rate=LR,
        num_leaves=3,
        min_child_samples=50,
        max_bin=511,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.70,
        reg_alpha=2.0,
        reg_lambda=10.0,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )


def build_cat() -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=N_EST,
        learning_rate=0.03 if USE_GPU else 0.08,
        depth=4,
        l2_leaf_reg=10.0,
        border_count=254,
        max_ctr_complexity=4 if USE_GPU else 1,
        eval_metric="AUC",
        task_type="GPU" if USE_GPU else "CPU",
        thread_count=-1,
        random_seed=42,
        verbose=0,
    )


DIGIT_EMB_SUFFIXES = ("__cents", "__tens", "__ones", "__lead", "__tenths", "__hundredths")


def _encode_digit_embeddings(df: pd.DataFrame) -> tuple[np.ndarray, list[int]]:
    """Trích xuất các Digit Keys có cardinality thấp/trung bình làm đầu vào nn.Embedding."""
    all_keys = _build_digit_keys(df)
    selected = sorted(k for k in all_keys if k.endswith(DIGIT_EMB_SUFFIXES))
    cols_arr = []
    cardinalities = []
    for k in selected:
        raw = all_keys[k]
        max_val = 99 if k.endswith("__cents") else 9
        # Map giá trị hợp lệ [0..max_val] -> [1..max_val+1]; giá trị biên/NaN -> 0
        valid = (raw >= 0) & (raw <= max_val)
        encoded = np.where(valid, raw + 1, 0).astype(np.int64)
        cols_arr.append(encoded)
        cardinalities.append(max_val + 2)
    return np.column_stack(cols_arr), cardinalities


if HAS_TORCH:
    class PeriodicEmbeddings(nn.Module):
        """PLR nhẹ cho biến số liên tục: sin/cos với tần số học được + chiếu tuyến tính từng cột."""
        def __init__(self, n_features: int, n_bins: int = 16, d_emb: int = 8, sigma: float = 0.1):
            super().__init__()
            self.freqs = nn.Parameter(torch.randn(n_features, n_bins) * sigma)
            self.proj_w = nn.Parameter(torch.randn(n_features, n_bins * 2, d_emb) * (1.0 / np.sqrt(n_bins * 2)))
            self.proj_b = nn.Parameter(torch.zeros(n_features, d_emb))
            self.act = nn.SiLU()

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            # x: (B, F) -> phases: (B, F, n_bins)
            v = 2.0 * np.pi * x.unsqueeze(-1) * self.freqs.unsqueeze(0)
            periodic = torch.cat([torch.sin(v), torch.cos(v)], dim=-1)
            emb = torch.einsum("bfk,fkd->bfd", periodic, self.proj_w) + self.proj_b.unsqueeze(0)
            return self.act(emb).flatten(1)

    class ResBlock(nn.Module):
        def __init__(self, dim: int, dropout: float = 0.2):
            super().__init__()
            self.block = nn.Sequential(
                nn.LayerNorm(dim),
                nn.SiLU(),
                nn.Linear(dim, dim),
                nn.LayerNorm(dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(dim, dim),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x + self.block(x)

    class WideAndDeepTabular(nn.Module):
        """Wide & Deep + PLR + Digit-Key Embeddings giữ nguyên log-odds cho các cột te_*."""
        def __init__(
            self,
            n_cont: int,
            n_te: int,
            n_bool: int,
            emb_cardinalities: list[int],
            width: int = 256,
            plr_dim: int = 8,
        ):
            super().__init__()
            n_dense = n_cont + n_te + n_bool
            # Wide (nhánh cộng tính tuyến tính trực tiếp)
            self.wide_linear = nn.Linear(n_dense, 1)
            self.wide_embs = nn.ModuleList([nn.Embedding(card, 1) for card in emb_cardinalities])

            # Deep (PLR cho cont + log-odds TE + bool + Cat/Digit Embeddings + Residual)
            self.plr = PeriodicEmbeddings(n_cont, n_bins=16, d_emb=plr_dim)
            self.deep_embs = nn.ModuleList([
                nn.Embedding(card, min(16, max(4, (card + 1) // 2)))
                for card in emb_cardinalities
            ])
            deep_in_dim = (n_cont * plr_dim) + n_te + n_bool + sum(e.embedding_dim for e in self.deep_embs)
            self.in_proj = nn.Linear(deep_in_dim, width)
            self.res1 = ResBlock(width, dropout=0.2)
            self.res2 = ResBlock(width, dropout=0.2)
            self.head = nn.Sequential(
                nn.LayerNorm(width),
                nn.SiLU(),
                nn.Linear(width, 1),
            )

        def forward(
            self,
            x_cont: torch.Tensor,
            x_te: torch.Tensor,
            x_bool: torch.Tensor,
            x_emb: torch.Tensor,
        ) -> torch.Tensor:
            x_dense = torch.cat([x_cont, x_te, x_bool], dim=1)
            wide_logit = self.wide_linear(x_dense).squeeze(1)
            for i, w_emb in enumerate(self.wide_embs):
                wide_logit = wide_logit + w_emb(x_emb[:, i]).squeeze(1)

            emb_outs = [emb(x_emb[:, i]) for i, emb in enumerate(self.deep_embs)]
            x_plr = self.plr(x_cont)
            deep_in = torch.cat([x_plr, x_te, x_bool, *emb_outs], dim=1)
            h = self.res2(self.res1(self.in_proj(deep_in)))
            deep_logit = self.head(h).squeeze(1)
            return wide_logit + deep_logit


def train_predict_nn(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_val: pd.DataFrame, y_val: pd.Series,
    X_test: pd.DataFrame,
    cat_cols: list[str],
    epochs: int = 25,
    batch_size: int = 2048,
    seeds: tuple[int, ...] = (42,),
) -> tuple[np.ndarray, np.ndarray]:
    """Huấn luyện Wide & Deep (logit-TE + Digit Embeddings + PLR) với multi-seed averaging trên không gian logit."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tr_nn = add_nn_extra_features(X_tr)
    X_val_nn = add_nn_extra_features(X_val)
    X_te_nn = add_nn_extra_features(X_test)

    # 1. Tách riêng cột te_* -> logit(clip(te)) + StandardScaler (bảo toàn thang đo log-odds)
    te_cols = [c for c in X_tr_nn.columns if c.startswith("te_")]
    cont_cols = [
        c for c in X_tr_nn.columns
        if c not in cat_cols and c not in BOOL_COLS and not c.startswith("te_")
    ]

    cont_imp = SimpleImputer(strategy="median")
    cont_scaler = QuantileTransformer(n_quantiles=2000, output_distribution="normal", subsample=150000, random_state=42)
    X_tr_cont = cont_scaler.fit_transform(cont_imp.fit_transform(X_tr_nn[cont_cols])).astype(np.float32)
    X_val_cont = cont_scaler.transform(cont_imp.transform(X_val_nn[cont_cols])).astype(np.float32)
    X_te_cont = cont_scaler.transform(cont_imp.transform(X_te_nn[cont_cols])).astype(np.float32)

    te_imp = SimpleImputer(strategy="median")
    te_scaler = StandardScaler()
    tr_te_logit = logit(np.clip(te_imp.fit_transform(X_tr_nn[te_cols]), 1e-6, 1.0 - 1e-6))
    val_te_logit = logit(np.clip(te_imp.transform(X_val_nn[te_cols]), 1e-6, 1.0 - 1e-6))
    te_te_logit = logit(np.clip(te_imp.transform(X_te_nn[te_cols]), 1e-6, 1.0 - 1e-6))
    X_tr_te = te_scaler.fit_transform(tr_te_logit).astype(np.float32)
    X_val_te = te_scaler.transform(val_te_logit).astype(np.float32)
    X_te_te = te_scaler.transform(te_te_logit).astype(np.float32)

    X_tr_bool = X_tr_nn[BOOL_COLS].to_numpy(dtype=np.float32)
    X_val_bool = X_val_nn[BOOL_COLS].to_numpy(dtype=np.float32)
    X_te_bool = X_te_nn[BOOL_COLS].to_numpy(dtype=np.float32)

    # 2. Gộp CAT_COLS + Digit Keys cardinality thấp/trung bình vào nn.Embedding
    def encode_cats(df_part: pd.DataFrame) -> np.ndarray:
        return np.column_stack([
            np.clip(df_part[c].cat.codes.to_numpy(dtype=np.int64) + 1, 0, len(X_tr_nn[c].cat.categories))
            for c in cat_cols
        ])

    cat_cards = [len(X_tr_nn[c].cat.categories) + 1 for c in cat_cols]
    X_tr_dig, dig_cards = _encode_digit_embeddings(X_tr_nn)
    X_val_dig, _ = _encode_digit_embeddings(X_val_nn)
    X_te_dig, _ = _encode_digit_embeddings(X_te_nn)

    X_tr_emb = np.hstack([encode_cats(X_tr_nn), X_tr_dig])
    X_val_emb = np.hstack([encode_cats(X_val_nn), X_val_dig])
    X_te_emb = np.hstack([encode_cats(X_te_nn), X_te_dig])
    emb_cards = cat_cards + dig_cards

    val_cont_t = torch.from_numpy(X_val_cont).to(device)
    val_te_t = torch.from_numpy(X_val_te).to(device)
    val_bool_t = torch.from_numpy(X_val_bool).to(device)
    val_emb_t = torch.from_numpy(X_val_emb).to(device)
    y_tr_t = torch.from_numpy(y_tr.to_numpy(dtype=np.float32))

    val_logit_sum = np.zeros(len(X_val), dtype=np.float64)
    te_logit_sum = np.zeros(len(X_test), dtype=np.float64)

    for seed in seeds:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        g = torch.Generator().manual_seed(seed)
        tr_loader = DataLoader(
            TensorDataset(
                torch.from_numpy(X_tr_cont),
                torch.from_numpy(X_tr_te),
                torch.from_numpy(X_tr_bool),
                torch.from_numpy(X_tr_emb),
                y_tr_t,
            ),
            batch_size=batch_size,
            shuffle=True,
            generator=g,
        )

        model = WideAndDeepTabular(
            n_cont=X_tr_cont.shape[1],
            n_te=X_tr_te.shape[1],
            n_bool=X_tr_bool.shape[1],
            emb_cardinalities=emb_cards,
            width=256,
            plr_dim=8,
        ).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        loss_fn = nn.BCEWithLogitsLoss()

        best_auc, best_state = -1.0, None
        for _ in range(epochs):
            model.train()
            for xb_c, xb_te, xb_b, xb_e, yb in tr_loader:
                xb_c, xb_te = xb_c.to(device), xb_te.to(device)
                xb_b, xb_e, yb = xb_b.to(device), xb_e.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb_c, xb_te, xb_b, xb_e), yb)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            with torch.no_grad():
                val_logits = model(val_cont_t, val_te_t, val_bool_t, val_emb_t).cpu().numpy()
            auc = roc_auc_score(y_val, expit(val_logits))
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        with torch.no_grad():
            val_logit_sum += model(val_cont_t, val_te_t, val_bool_t, val_emb_t).cpu().numpy() / len(seeds)
            te_seed_logits = []
            for i in range(0, len(X_te_cont), 16384):
                te_c = torch.from_numpy(X_te_cont[i:i + 16384]).to(device)
                te_t = torch.from_numpy(X_te_te[i:i + 16384]).to(device)
                te_b = torch.from_numpy(X_te_bool[i:i + 16384]).to(device)
                te_e = torch.from_numpy(X_te_emb[i:i + 16384]).to(device)
                te_seed_logits.append(model(te_c, te_t, te_b, te_e).cpu().numpy())
            te_logit_sum += np.concatenate(te_seed_logits) / len(seeds)

    return expit(val_logit_sum), expit(te_logit_sum)

In [ ]:
# 1. Đọc dữ liệu & đồng bộ category
train = add_features(pd.read_csv(DATA_DIR / "train.csv"))
test = add_features(pd.read_csv(DATA_DIR / "test.csv"))

for col in CAT_COLS:
    cats = pd.CategoricalDtype(categories=sorted(set(train[col].dropna()).union(test[col].dropna())))
    train[col] = train[col].astype(cats)
    test[col] = test[col].astype(cats)

X = train.drop(columns=["id", "is_fraud"])
y = train["is_fraud"].astype(int)
X_test = test.drop(columns=["id"])

# 2. Stratified 5-Fold Cross-Validation (tính trung bình test theo không gian logit)
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = {"xgb": np.zeros(len(X)), "lgb": np.zeros(len(X))}
test_logits = {"xgb": np.zeros(len(X_test)), "lgb": np.zeros(len(X_test))}
if INCLUDE_CATBOOST:
    oof_preds["cat"] = np.zeros(len(X))
    test_logits["cat"] = np.zeros(len(X_test))
if INCLUDE_NN:
    oof_preds["nn"] = np.zeros(len(X))
    test_logits["nn"] = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    t0 = time.time()
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    X_tr, X_val, X_te_fold = add_fold_digit_te(
        X.iloc[tr_idx], y_tr, X.iloc[val_idx], X_test, smooth=20.0
    )

    # --- XGBoost ---
    xgb_model = build_xgb()
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_preds["xgb"][val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_logits["xgb"] += logit(np.clip(xgb_model.predict_proba(X_te_fold)[:, 1], 1e-6, 1.0 - 1e-6)) / n_splits

    # --- LightGBM ---
    lgb_model = build_lgb()
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)],
    )
    oof_preds["lgb"][val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_logits["lgb"] += logit(np.clip(lgb_model.predict_proba(X_te_fold)[:, 1], 1e-6, 1.0 - 1e-6)) / n_splits

    parts = [
        f"Fold {fold}/{n_splits} ({time.time() - t0:.1f}s)",
        f"XGB={roc_auc_score(y_val, oof_preds['xgb'][val_idx]):.5f}",
        f"LGB={roc_auc_score(y_val, oof_preds['lgb'][val_idx]):.5f}",
    ]

    # --- CatBoost ---
    if INCLUDE_CATBOOST:
        cat_model = build_cat()
        cat_model.fit(X_tr, y_tr, cat_features=CAT_COLS, eval_set=(X_val, y_val), early_stopping_rounds=EARLY_STOP)
        oof_preds["cat"][val_idx] = cat_model.predict_proba(X_val)[:, 1]
        test_logits["cat"] += logit(np.clip(cat_model.predict_proba(X_te_fold)[:, 1], 1e-6, 1.0 - 1e-6)) / n_splits
        parts.append(f"CAT={roc_auc_score(y_val, oof_preds['cat'][val_idx]):.5f}")

    # --- PyTorch Wide & Deep NN v2 ---
    if INCLUDE_NN:
        val_nn, te_nn = train_predict_nn(
            X_tr, y_tr, X_val, y_val, X_te_fold, CAT_COLS,
            epochs=25, batch_size=2048, seeds=NN_SEEDS,
        )
        oof_preds["nn"][val_idx] = val_nn
        test_logits["nn"] += logit(np.clip(te_nn, 1e-6, 1.0 - 1e-6)) / n_splits
        parts.append(f"NN={roc_auc_score(y_val, val_nn):.5f}")

    print(" | ".join(parts))

test_preds = {k: expit(v) for k, v in test_logits.items()}

In [ ]:
# 3. Lưu OOF / Test .npy & chạy Nested-CV Meta-Stacking trên không gian logit
names = list(oof_preds.keys())
for name in names:
    print(f"OOF {name.upper():4s}: {roc_auc_score(y, oof_preds[name]):.5f}")
    np.save(OUT_DIR / f"oof_{name}.npy", oof_preds[name].astype(np.float64))
    np.save(OUT_DIR / f"test_{name}.npy", test_preds[name].astype(np.float64))

oof_mat = np.column_stack([oof_preds[n] for n in names])
oof_logit_mat = logit(np.clip(oof_mat, 1e-6, 1.0 - 1e-6))
rank_mat = np.column_stack([rankdata(oof_preds[n]) / len(y) for n in names])
test_mat = np.column_stack([test_preds[n] for n in names])
test_logit_mat = np.column_stack([test_logits[n] for n in names])
test_rank_mat = np.column_stack([rankdata(test_preds[n]) / len(X_test) for n in names])
y_arr = y.to_numpy(dtype=np.float64)


def fit_slsqp_logit(X_l: np.ndarray, y_t: np.ndarray) -> np.ndarray:
    m = X_l.shape[1]
    def obj_and_grad(w: np.ndarray) -> tuple[float, np.ndarray]:
        p = np.clip(expit(X_l @ w), 1e-12, 1.0 - 1e-12)
        loss = -np.mean(y_t * np.log(p) + (1.0 - y_t) * np.log(1.0 - p))
        grad = (X_l.T @ (p - y_t)) / len(y_t)
        return float(loss), grad
    res = minimize(
        obj_and_grad,
        np.full(m, 1.0 / m),
        jac=True,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * m,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    )
    w = np.clip(res.x, 0.0, 1.0)
    return w / np.sum(w)


def fit_hill_climbing(X_l: np.ndarray, y_t: np.ndarray, n_steps: int = 40) -> np.ndarray:
    m = X_l.shape[1]
    if len(y_t) > 150000:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=150000, random_state=42)
        idx, _ = next(sss.split(X_l, y_t))
        X_sub, y_sub = X_l[idx], y_t[idx]
    else:
        X_sub, y_sub = X_l, y_t

    counts = np.zeros(m, dtype=np.float64)
    best_idx = int(np.argmax([roc_auc_score(y_sub, X_sub[:, j]) for j in range(m)]))
    counts[best_idx] += 1.0
    cur_sum = X_sub[:, best_idx].copy()
    cur_auc = roc_auc_score(y_sub, cur_sum)

    for step in range(2, n_steps + 1):
        best_j, best_step_auc = -1, cur_auc
        for j in range(m):
            cand = (cur_sum + X_sub[:, j]) / step
            auc = roc_auc_score(y_sub, cand)
            if auc > best_step_auc + 1e-7:
                best_step_auc = auc
                best_j = j
        if best_j == -1:
            break
        counts[best_j] += 1.0
        cur_sum = cur_sum + X_sub[:, best_j]
        cur_auc = best_step_auc
    return counts / np.sum(counts)


# Nested-CV trên 5 fold riêng biệt để đánh giá Meta-Learner không bị overfit trọng số
meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
meta_oof = {
    "slsqp_logit": np.zeros(len(y_arr)),
    "l2_logreg": np.zeros(len(y_arr)),
    "hill_climb_logit": np.zeros(len(y_arr)),
}
meta_test_logits = {k: np.zeros(len(X_test)) for k in meta_oof}

for tr_i, val_i in meta_skf.split(oof_logit_mat, y_arr):
    X_m_tr, y_m_tr = oof_logit_mat[tr_i], y_arr[tr_i]
    X_m_val = oof_logit_mat[val_i]

    # 1. Constrained SLSQP
    w_slsqp = fit_slsqp_logit(X_m_tr, y_m_tr)
    meta_oof["slsqp_logit"][val_i] = expit(X_m_val @ w_slsqp)
    meta_test_logits["slsqp_logit"] += (test_logit_mat @ w_slsqp) / 5.0

    # 2. L2 LogisticRegression
    lr_meta = LogisticRegression(C=1.0, max_iter=1000)
    lr_meta.fit(X_m_tr, y_m_tr)
    meta_oof["l2_logreg"][val_i] = lr_meta.predict_proba(X_m_val)[:, 1]
    meta_test_logits["l2_logreg"] += lr_meta.decision_function(test_logit_mat) / 5.0

    # 3. Hill Climbing
    w_hc = fit_hill_climbing(X_m_tr, y_m_tr, n_steps=40)
    meta_oof["hill_climb_logit"][val_i] = expit(X_m_val @ w_hc)
    meta_test_logits["hill_climb_logit"] += (test_logit_mat @ w_hc) / 5.0

print("\n--- Nested-CV Meta-Stacker Comparison ---")
best_meta_name, best_meta_auc = "", -1.0
for m_name, m_oof in meta_oof.items():
    m_auc = roc_auc_score(y_arr, m_oof)
    print(f"Meta {m_name:18s}: OOF ROC-AUC = {m_auc:.5f}")
    if m_auc > best_meta_auc:
        best_meta_auc = m_auc
        best_meta_name = m_name

# Đối chiếu thêm chuẩn 3 không gian cố định (có sửa lỗi dòng 515 khi best_mode == 'rank')
w_eq = np.full(len(names), 1.0 / len(names))
space_refs = {
    "prob": (oof_mat @ w_eq, test_mat @ w_eq),
    "logit": (expit(oof_logit_mat @ w_eq), expit(test_logit_mat @ w_eq)),
    "rank": (rank_mat @ w_eq, test_rank_mat @ w_eq),
}
for sp_mode, (sp_oof, _) in space_refs.items():
    print(f"Uniform {sp_mode:15s}: OOF ROC-AUC = {roc_auc_score(y_arr, sp_oof):.5f}")

# In trọng số refit trên 100% OOF để quan sát đóng góp từng mô hình
print("\nFull-OOF SLSQP weights:", ", ".join(f"{n.upper()}={w:.3f}" for n, w in zip(names, fit_slsqp_logit(oof_logit_mat, y_arr))))
print("Full-OOF HillClimb weights:", ", ".join(f"{n.upper()}={w:.3f}" for n, w in zip(names, fit_hill_climbing(oof_logit_mat, y_arr))))

oof_ens = meta_oof[best_meta_name]
test_ens = expit(meta_test_logits[best_meta_name])
print(f"\nSelected Meta-Learner: {best_meta_name} | Nested-CV OOF ROC-AUC: {best_meta_auc:.5f}")
print(classification_report(y, (oof_ens >= 0.5).astype(int)))

submission = pd.DataFrame({
    "id": test["id"],
    "is_fraud": test_ens,
})
submission.to_csv("submission.csv", index=False)
print("Đã lưu submission.csv:", submission.shape)

In [ ]:
# 4. Phân tích lỗi, Ma trận tương quan & Chỉ số báo cáo (Calibration, PR-AUC, Brier, Recall@FPR=0.5%)
def recall_at_fpr(y_true: np.ndarray, y_prob: np.ndarray, max_fpr: float = 0.005) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    mask = fpr <= max_fpr
    return float(tpr[mask][-1]) if np.any(mask) else 0.0


eval_dict = {n.upper(): oof_preds[n] for n in names}
eval_dict[f"STACK ({best_meta_name})"] = oof_ens

rows = []
for model_label, probs in eval_dict.items():
    rows.append({
        "Model": model_label,
        "ROC-AUC": roc_auc_score(y_arr, probs),
        "PR-AUC": average_precision_score(y_arr, probs),
        "Brier": brier_score_loss(y_arr, probs),
        "Recall@FPR=0.5%": recall_at_fpr(y_arr, probs, 0.005),
    })
metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False, float_format=lambda v: f"{v:.5f}"))

print("\nMa trận tương quan dự đoán (OOF Logit):")
corr_df = pd.DataFrame(oof_logit_mat, columns=[n.upper() for n in names]).corr()
print(corr_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ 1: Reliability Curve (Calibration)
for model_label, probs in eval_dict.items():
    frac_pos, mean_pred = calibration_curve(y_arr, probs, n_bins=15, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", linewidth=1.6, label=model_label)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect")
axes[0].set_title("Reliability Curve (Quantile Bins)")
axes[0].set_xlabel("Mean Predicted Probability")
axes[0].set_ylabel("Fraction of Positives")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Biểu đồ 2: Heatmap tương quan giữa các Base Models
im = axes[1].imshow(corr_df.values, vmin=0.8, vmax=1.0, cmap="YlGnBu")
axes[1].set_xticks(range(len(corr_df.columns)))
axes[1].set_yticks(range(len(corr_df.columns)))
axes[1].set_xticklabels(corr_df.columns)
axes[1].set_yticklabels(corr_df.columns)
for i in range(len(corr_df)):
    for j in range(len(corr_df)):
        axes[1].text(j, i, f"{corr_df.values[i, j]:.3f}", ha="center", va="center", color="black")
axes[1].set_title("OOF Logit Correlation Matrix")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(OUT_DIR / "diagnostics.png", dpi=150)
plt.show()